# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedaamar744-ux/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
### Signal checks

**Signal 1 — Volume: CONFIRMED.** Higher-impression content has substantially greater search visibility, making impressions a useful signal for prioritizing content with meaningful search opportunity.

**Signal 2 — CTR vs Position: CONFIRMED.** Observed CTR decreases as search position worsens: 0.381% for positions 1–3, 0.324% for positions 4–10, 0.315% for positions 11–20, and 0.131% for positions 20+.

### Baseline rule

Prioritize content that has meaningful search visibility but weak click performance, while using search position as context.

### Reason codes

- `HIGH_VOLUME_LOW_CTR` — High search impressions with weak click-through performance.
- `HIGH_VOLUME_WEAK_POSITION` — High search impressions with a weaker search position.
- `HIGH_VOLUME` — High search visibility without another stronger signal.
- `LOW_PRIORITY_SIGNAL` — A lower-priority item with only one supporting signal.
- `LOW_VOLUME` — Low search visibility; lower priority for the baseline queue.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

# Load Hugging Face token
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect to DuckDB
con = duckdb.connect()

# Authenticate with Hugging Face
con.execute("""
CREATE SECRET IF NOT EXISTS (
    TYPE huggingface,
    TOKEN ?
)
""", [HF_TOKEN])

# March 2026 warehouse data
path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# Load only the columns needed for Week 4
df = con.execute(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    gsc_data_available
FROM read_parquet('{path}')
WHERE gsc_data_available = TRUE
""").df()

# Convert date
df["report_date"] = pd.to_datetime(df["report_date"])

# Calculate CTR
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    np.nan
)

print("Rows:", len(df))
print("Date range:", df["report_date"].min(), "→", df["report_date"].max())
print("Clients:", df["client_hash_id"].nunique())
print("Content items:", df["content_hash_id"].nunique())

print("\nMissing values:")
print(df[[
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ctr"
]].isna().sum())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 3611061
Date range: 2026-03-01 00:00:00 → 2026-03-31 00:00:00
Clients: 47
Content items: 176738

Missing values:
gsc_impressions     0
gsc_clicks          0
gsc_avg_position    0
ctr                 0
dtype: int64


In [2]:
# Signal 1: Volume (GSC impressions)

volume_df = df[df["gsc_impressions"] > 0].copy()

volume_df["volume_bucket"] = pd.qcut(
    volume_df["gsc_impressions"],
    q=4,
    duplicates="drop"
)

volume_table = (
    volume_df
    .groupby("volume_bucket", observed=True)
    .agg(
        n=("gsc_impressions", "size"),
        median_impressions=("gsc_impressions", "median"),
        median_ctr=("ctr", "median"),
        median_position=(
            "gsc_avg_position",
            lambda x: x[x > 0].median()
        )
    )
    .reset_index()
)

display(volume_table)

,volume_bucket,n,median_impressions,median_ctr,median_position
0,"(0.999, 4.0]",973112,2.0,0.0,9.000000
1,"(4.0, 16.0]",868861,9.0,0.0,10.000000
2,"(16.0, 62.0]",873000,31.0,0.0,7.565217
3,"(62.0, 40084.0]",896088,150.0,0.0,5.909091


In [3]:
# Signal 2: CTR vs Position

position_df = df[
    (df["gsc_avg_position"] > 0) &
    (df["gsc_impressions"] > 0)
].copy()

# Use interpretable Google position bands
position_df["position_bucket"] = pd.cut(
    position_df["gsc_avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "20+"],
    include_lowest=True
)

position_table = (
    position_df
    .groupby("position_bucket", observed=True)
    .agg(
        n=("gsc_avg_position", "size"),
        median_position=("gsc_avg_position", "median"),
        median_ctr=("ctr", "median"),
        median_impressions=("gsc_impressions", "median")
    )
    .reset_index()
)

display(position_table)

,position_bucket,n,median_position,median_ctr,median_impressions
0,1-3,564173,1.935484,0.0,23.0
1,4-10,1456122,5.868601,0.0,23.0
2,11-20,519223,14.000000,0.0,20.0
3,20+,908354,37.086957,0.0,9.0


In [4]:
# Check the CTR distribution before building the baseline

ctr_check = (
    position_df
    .assign(has_clicks=position_df["gsc_clicks"] > 0)
    .groupby("position_bucket", observed=True)
    .agg(
        n=("gsc_clicks", "size"),
        rows_with_clicks=("has_clicks", "sum"),
        total_clicks=("gsc_clicks", "sum"),
        total_impressions=("gsc_impressions", "sum")
    )
    .reset_index()
)

ctr_check["observed_ctr"] = (
    ctr_check["total_clicks"] /
    ctr_check["total_impressions"]
)

display(ctr_check)

,position_bucket,n,rows_with_clicks,total_clicks,total_impressions,observed_ctr
0,1-3,564173,91485,204283,53559848,0.003814
1,4-10,1456122,221682,445828,137830113,0.003235
2,11-20,519223,53551,92449,29386006,0.003146
3,20+,908354,50112,78098,59412876,0.001314


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*
### Baseline scoring rule

The ranked queue uses a transparent, hand-written score based on March 31 observations.

* **+2 points:** high search impressions
* **+1 point:** low CTR
* **+1 point:** weak search position (>10)

Items with higher scores are prioritized for review. The queue is ranked by baseline score first, then by search impressions as a tie-breaker.

The score is a decision-support rule, not a trained model, and uses only information available within the March 2026 development window.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the baseline ranked queue

queue = df.copy()

# Use the latest available day in March as the decision snapshot
decision_date = queue["report_date"].max()

queue = queue[queue["report_date"] == decision_date].copy()

# Volume threshold: top 25% by impressions
volume_threshold = queue["gsc_impressions"].quantile(0.75)

# CTR threshold: bottom 25% among rows with impressions
ctr_threshold = queue.loc[
    queue["gsc_impressions"] > 0,
    "ctr"
].quantile(0.25)

# Position threshold: worse than position 10
position_threshold = 10

# Signal flags
queue["high_volume"] = queue["gsc_impressions"] >= volume_threshold
queue["low_ctr"] = queue["ctr"] <= ctr_threshold
queue["weak_position"] = (
    (queue["gsc_avg_position"] > 0) &
    (queue["gsc_avg_position"] > position_threshold)
)

# Transparent baseline score
queue["baseline_score"] = (
    queue["high_volume"].astype(int) * 2
    + queue["low_ctr"].astype(int) * 1
    + queue["weak_position"].astype(int) * 1
)

# Reason code
queue["reason_code"] = np.select(
    [
        queue["high_volume"] & queue["low_ctr"],
        queue["high_volume"] & queue["weak_position"],
        queue["high_volume"],
        queue["low_ctr"] | queue["weak_position"]
    ],
    [
        "HIGH_VOLUME_LOW_CTR",
        "HIGH_VOLUME_WEAK_POSITION",
        "HIGH_VOLUME",
        "LOW_PRIORITY_SIGNAL"
    ],
    default="LOW_VOLUME"
)

# Action label
queue["action"] = np.where(
    queue["baseline_score"] >= 3,
    "REVIEW_FOR_REFRESH",
    np.where(
        queue["baseline_score"] >= 2,
        "REVIEW",
        "LOW_PRIORITY"
    )
)

# Rank
queue = queue.sort_values(
    ["baseline_score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

# Keep the decision-support fields
output = queue[
    [
        "rank",
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "baseline_score",
        "action",
        "reason_code",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position"
    ]
].copy()

# Write the required CSV
output_path = "work/outputs/baseline_action_score.csv"

import os
os.makedirs("work/outputs", exist_ok=True)

output.to_csv(output_path, index=False)

print("Decision date:", decision_date)
print("Volume threshold:", volume_threshold)
print("CTR threshold:", ctr_threshold)
print("Rows ranked:", len(output))
print("CSV written to:", output_path)

display(output.head(20))

Decision date: 2026-03-31 00:00:00
Volume threshold: 64.0
CTR threshold: 0.0
Rows ranked: 125222
CSV written to: work/outputs/baseline_action_score.csv


,rank,report_date,client_hash_id,content_hash_id,baseline_score,action,reason_code,gsc_impressions,gsc_clicks,ctr,gsc_avg_position
0,1,2026-03-31,client_23a62021009f63c4,content_73aa61dcedebbf30,4,REVIEW_FOR_REFRESH,HIGH_VOLUME_LOW_CTR,5521,0,0.0,47.821953
1,2,2026-03-31,client_23a62021009f63c4,content_6530fa9d297c46eb,4,REVIEW_FOR_REFRESH,HIGH_VOLUME_LOW_CTR,5364,0,0.0,89.848248
2,3,2026-03-31,client_20259bd6705d81d4,content_82e35c4845e6c391,4,REVIEW_FOR_REFRESH,HIGH_VOLUME_LOW_CTR,4922,0,0.0,31.245632
3,4,2026-03-31,client_23a62021009f63c4,content_a3a1317f7c2bc3dd,4,REVIEW_FOR_REFRESH,HIGH_VOLUME_LOW_CTR,3601,0,0.0,35.579006
4,5,2026-03-31,client_23a62021009f63c4,content_bdf60c86117079be,4,REVIEW_FOR_REFRESH,HIGH_VOLUME_LOW_CTR,3580,0,0.0,29.949721
5,6,2026-03-31,client_23a62021009f63c4,content_fcd637c7229d1e3c,4,REVIEW_FOR_REFRESH,HIGH_VOLUME_LOW_CTR,3492,0,0.0,40.768614
6,7,2026-03-31,client_23a62021009f63c4,content_f6723f0229e1bfdc,4,REVIEW_FOR_REFRESH,HIGH_VOLUME_LOW_CTR,3273,0,0.0,16.384357
7,8,2026-03-31,client_23a62021009f63c4,content_164c1f53f13bcee1,4,REVIEW_FOR_REFRESH,HIGH_VOLUME_LOW_CTR,3255,0,0.0,20.241167
8,9,2026-03-31,client_23a62021009f63c4,content_4889078a4e2655ff,4,REVIEW_FOR_REFRESH,HIGH_VOLUME_LOW_CTR,3219,0,0.0,31.531532
9,10,2026-03-31,client_20259bd6705d81d4,content_9e8c3b7e2f8a1093,4,REVIEW_FOR_REFRESH,HIGH_VOLUME_LOW_CTR,3074,0,0.0,25.329538


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
### Top-20 review

For each of the top 20 items, review the selected action, reason code, confidence note, and what could make the recommendation wrong.

The review is based only on the observed March 31 signals used by the baseline. It is decision-support, not a claim that every selected item definitely requires a refresh.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Top-20 review

top20 = output.head(20).copy()

top20["confidence_note"] = np.select(
    [
        top20["baseline_score"] == 4,
        top20["baseline_score"] == 3,
        top20["baseline_score"] == 2
    ],
    [
        "Higher confidence: multiple observed signals agree.",
        "Moderate confidence: strong volume plus one supporting signal.",
        "Lower confidence: only one main signal is present."
    ],
    default="Low confidence: limited supporting evidence."
)

top20["what_would_make_it_wrong"] = np.select(
    [
        top20["reason_code"] == "HIGH_VOLUME_LOW_CTR",
        top20["reason_code"] == "HIGH_VOLUME_WEAK_POSITION",
        top20["reason_code"] == "HIGH_VOLUME"
    ],
    [
        "Clicks or impressions may be affected by tracking/data-quality issues, or the content may have low click intent despite high visibility.",
        "The position may reflect a temporary query mix or measurement condition rather than a refresh opportunity.",
        "High impressions alone may not indicate a refresh need; the content may already be performing appropriately for its search demand."
    ],
    default="The observed signal may not represent a genuine refresh opportunity."
)

review = top20[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position"
    ]
].copy()

display(review)

,rank,client_hash_id,content_hash_id,action,reason_code,confidence_note,what_would_make_it_wrong,gsc_impressions,gsc_clicks,ctr,gsc_avg_position
0,1,client_23a62021009f63c4,content_73aa61dcedebbf30,REVIEW_FOR_REFRESH,HIGH_VOLUME_LOW_CTR,Higher confidence: multiple observed signals a...,Clicks or impressions may be affected by track...,5521,0,0.0,47.821953
1,2,client_23a62021009f63c4,content_6530fa9d297c46eb,REVIEW_FOR_REFRESH,HIGH_VOLUME_LOW_CTR,Higher confidence: multiple observed signals a...,Clicks or impressions may be affected by track...,5364,0,0.0,89.848248
2,3,client_20259bd6705d81d4,content_82e35c4845e6c391,REVIEW_FOR_REFRESH,HIGH_VOLUME_LOW_CTR,Higher confidence: multiple observed signals a...,Clicks or impressions may be affected by track...,4922,0,0.0,31.245632
3,4,client_23a62021009f63c4,content_a3a1317f7c2bc3dd,REVIEW_FOR_REFRESH,HIGH_VOLUME_LOW_CTR,Higher confidence: multiple observed signals a...,Clicks or impressions may be affected by track...,3601,0,0.0,35.579006
4,5,client_23a62021009f63c4,content_bdf60c86117079be,REVIEW_FOR_REFRESH,HIGH_VOLUME_LOW_CTR,Higher confidence: multiple observed signals a...,Clicks or impressions may be affected by track...,3580,0,0.0,29.949721
5,6,client_23a62021009f63c4,content_fcd637c7229d1e3c,REVIEW_FOR_REFRESH,HIGH_VOLUME_LOW_CTR,Higher confidence: multiple observed signals a...,Clicks or impressions may be affected by track...,3492,0,0.0,40.768614
6,7,client_23a62021009f63c4,content_f6723f0229e1bfdc,REVIEW_FOR_REFRESH,HIGH_VOLUME_LOW_CTR,Higher confidence: multiple observed signals a...,Clicks or impressions may be affected by track...,3273,0,0.0,16.384357
7,8,client_23a62021009f63c4,content_164c1f53f13bcee1,REVIEW_FOR_REFRESH,HIGH_VOLUME_LOW_CTR,Higher confidence: multiple observed signals a...,Clicks or impressions may be affected by track...,3255,0,0.0,20.241167
8,9,client_23a62021009f63c4,content_4889078a4e2655ff,REVIEW_FOR_REFRESH,HIGH_VOLUME_LOW_CTR,Higher confidence: multiple observed signals a...,Clicks or impressions may be affected by track...,3219,0,0.0,31.531532
9,10,client_20259bd6705d81d4,content_9e8c3b7e2f8a1093,REVIEW_FOR_REFRESH,HIGH_VOLUME_LOW_CTR,Higher confidence: multiple observed signals a...,Clicks or impressions may be affected by track...,3074,0,0.0,25.329538


## 4. Weak picks + leakage check


*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks

The main weak-pick pattern is the zero-CTR threshold. Because the lower CTR quartile is zero, high-impression items with zero clicks receive the strongest score. These items are plausible review candidates, but zero clicks alone does not prove that a refresh is needed.

Some top-ranked items have very weak search positions. This is a limitation of the baseline because poor position may indicate a broader visibility or query-mix problem rather than a simple refresh opportunity.

Leakage check

The baseline uses only March 2026 observations available at the decision date of March 31.

It does not use trend_pct, trend_direction, is_declining_label, future-window outcomes, client names, URLs, or supervised labels. Pseudonymous client and content IDs are used only for identification and grouping, not as model inputs.

The score is a transparent hand-written rule and does not use fitted model weights.

In [7]:
# Weak picks + leakage checks

print("=== Weak-pick checks ===")

print("Top-20 zero-click rows:",
      int((top20["gsc_clicks"] == 0).sum()),
      "of", len(top20))

print("Top-20 average impressions:",
      round(top20["gsc_impressions"].mean(), 2))

print("Top-20 average position:",
      round(top20["gsc_avg_position"].mean(), 2))

print("\n=== Leakage checks ===")

# Prohibited label-derived fields must not be present in the queue
prohibited_columns = {
    "trend_pct",
    "trend_direction",
    "is_declining_label"
}

found_prohibited = prohibited_columns.intersection(output.columns)

print("Label-derived columns in output:", found_prohibited)

# Confirm decision snapshot
print(
    "Decision date:",
    output["report_date"].min(),
    "→",
    output["report_date"].max()
)

# Confirm no future dates entered the queue
print(
    "Rows after March 31:",
    int((output["report_date"] > pd.Timestamp("2026-03-31")).sum())
)

# Confirm IDs are not used in the score itself
score_inputs = [
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_avg_position"
]

print("Score input columns:", score_inputs)

# Final pass/fail
leakage_ok = (
    len(found_prohibited) == 0
    and (output["report_date"] <= pd.Timestamp("2026-03-31")).all()
)

print("\nLeakage check:", "PASS" if leakage_ok else "REVIEW")

=== Weak-pick checks ===
Top-20 zero-click rows: 20 of 20
Top-20 average impressions: 3329.35
Top-20 average position: 32.97

=== Leakage checks ===
Label-derived columns in output: set()
Decision date: 2026-03-31 00:00:00 → 2026-03-31 00:00:00
Rows after March 31: 0
Score input columns: ['gsc_impressions', 'gsc_clicks', 'ctr', 'gsc_avg_position']

Leakage check: PASS


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.